# RFT Training：直接在 `gemma-4-31b-it-qat-w4a16-ct` 上训练 LoRA → submission.zip

实现 `TASK.md` §3（阶段一）的训练，当前方案见 `docs/plan_rft.md`。输入是 `gemma-4-rft-data.ipynb` 产出的 `train.jsonl`，输出是可以直接提交的 LoRA adapter 和 `submission.zip`。

**做法**
- **直接在比赛的 int4 权重上训练**：冻结 w4a16-ct 权重，在其上训练 LoRA。训练与推理看到的 base 权重完全相同，也不需要再挂 bf16 模型。
- **不用 peft / bitsandbytes / transformers 的 compressed-tensors 量化器**：
  - 新版 compressed-tensors 会在第一次 forward 时把整个模型解压成 bf16（62 GB，OOM）。
  - peft 不认识自定义量化层。
  - 离线镜像里也可能没有这些包。
  - 因此本 notebook 自己实现以下几部分：
    - `W4A16Linear`：自定义 autograd，即时反量化，backward 不保存 bf16 权重。
    - `LoRALinear`
    - PEFT 格式导出
- **多卡**：decoder 层按连续分组放到 4 张 L4 上，用 forward pre-hook 把输入搬到对应的卡，并开启 gradient checkpointing。logits 只在 loss 位置计算，省掉 16k × 262k 的约 17 GB。
- **简短而正确**：
  - 数据 notebook 已只保留通过测试、改到参考文件、首次编辑 ≤ 第 12 次调用，且预估耗时在时间预算内的轨迹。
  - 本 notebook 再做三件事：
    - 去掉 ADK 上下文压缩的 summarizer turn（没有 tool call，不是 agent 决策）。
    - 去掉生成超过 `MAX_LOSS_TOKENS` 的 turn（超长思考）。
    - 序列内按 token 加权平均：结束思考的 `<channel|>`、`<|tool_call>` 和工具名的权重为 `DECISION_WEIGHT`。
    - 序列间取加权平均：首次编辑 turn 的权重为 `FIRST_EDIT_WEIGHT`，其余为 1。
    - 留出 2–3 题不参加训练，用来挑选最后一个未过拟合的 checkpoint。

**挂载**
- 竞赛数据
- `google/gemma-4` 模型（`gemma-4-31b-it-qat-w4a16-ct`）
- wheelhouse
- RFT 数据：把 `rft_data_<RUN_NAME>/` 上传为 dataset，或挂 `gemma-4-rft-data.ipynb` 的 Output
- 你的 submission 数据集：不带 adapter，且必须与采样时的 submission 相同（指纹不一致会报错）

Accelerator 选 **GPU L4x4**，Internet **off**。

**运行**
1. 先设 `SMOKE = True` 跑一次，约 20–30 min：用 5 条序列训练 2 步，把加载 → 训练 → 导出 → 打包整条链路走一遍。产出 `submission_SMOKE.zip`（**不要提交**）。
2. 再设 `SMOKE = False`，用 *Save & Run All* 全量跑。约 4 h，`MAX_TRAIN_HOURS` 是上限，前几步跑完后会打印实测值。
3. 之后用 dev-eval notebook 挂新的 `submission.zip`，做 Gate 1（vLLM 加载 adapter、`/v1/models` 可见）和 Gate 2a。

**输出**（`/kaggle/working/`）
- `submission.zip`：原 submission + `adapters/main_lora/` + `agent.yaml` 里的 `adapter: main_lora`
- `submission_alpha32.zip`：同一份 adapter，`lora_alpha=32`，不需要额外训练
- `main_lora/`：`adapter_config.json`、`adapter_model.safetensors`
- `train_meta.json`：数据统计、超参、loss 曲线、探针结果
- `train_log.jsonl`：每个优化步一行


## 0. Run Config

In [ ]:
import time

NOTEBOOK_START = time.time()

# ================================ RUN CONFIG ================================
RUN_NAME = 'lora_p2'

# Directory holding train.jsonl + stats.json from gemma-4-rft-data.ipynb.
# None = auto-detect under /kaggle/input (a dataset upload of rft_data_<run>/, or the
# attached output of the data notebook; rft_data_*.zip is extracted if needed).
RFT_DATA_DIR = None
# The competition checkpoint. Training runs directly on these int4 weights (frozen).
MODEL_PATH = '/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2'
# Your submission: path to submission.zip or to a directory containing agent.yaml.
# None = auto-detect among the attached datasets. It must not ship an adapter yet.
SUBMISSION_INPUT = None
ADAPTER_NAME = 'main_lora'

# LoRA (TASK.md §3.2)
LORA_R = 16
LORA_ALPHA = 16                # alpha / r = 1.0
LORA_DROPOUT = 0.05
TARGET_MODULES = ('q_proj', 'k_proj', 'v_proj', 'o_proj')   # attention only, language-model layers only
# p1b used lr 5e-5 x 38 steps: max|lora_B| ended at the Adam movement bound (about 1e-3), so training
# was limited by steps x lr, not by convergence. These settings allow about 7.6 times more movement.
LR = 1e-4                      # cosine schedule, WARMUP_RATIO warmup
WARMUP_RATIO = 0.1
EPOCHS = 2
GRAD_ACCUM = 2                 # sequences per optimizer step (batch size 1 each)
MAX_GRAD_NORM = 1.0
WEIGHT_DECAY = 0.0
SEED = 0

# Data filter, for short and correct outputs (the data notebook already kept only
# efficient, trusted passes).
MAX_LOSS_TOKENS = 1536         # drop turns that generate more than this (very long thinking)
MAX_SEQ_TOKENS = 16384         # drop longer sequences (memory)
REQUIRE_TOOL_CALL = True       # drop turns without <|tool_call> (ADK compaction summaries)

# Loss weights. Plain imitation copies long thoughts; these target the decisions that save time.
DECISION_WEIGHT = 5.0          # <channel|> (stop thinking), <|tool_call> and the tool name, within a sequence
FIRST_EDIT_WEIGHT = 2.0        # sequences whose loss turn is the first source edit, within a batch

# Holdout: tasks kept out of training. Evaluated every SAVE_EVERY_STEPS; the adapter that gets exported
# is the last one whose weighted NLL is within HOLDOUT_TOL of the best so far.
HOLDOUT_MIN_TASKS = 8          # with fewer tasks there is no holdout (and a warning)
HOLDOUT_MAX_SEQS = 12          # shortest sequences of the holdout tasks (evaluation cost)
HOLDOUT_TOL = 0.005

# Memory / speed
ATTN_IMPL = 'sdpa'
COMPILE_DEQUANT = True         # torch.compile the int4 dequant; falls back to eager if it fails
GPU0_RESERVE_GB = 3.0          # GPU 0 also holds the embeddings and the logits at the loss positions
LAYER_SPLIT = None             # e.g. [8, 18, 18, 16] decoder layers per GPU; None = automatic

# Time guard: Kaggle stops sessions at 12 h. Training stops before this many hours
# since notebook start; the adapter is then exported and packaged as usual.
MAX_TRAIN_HOURS = 5.0
SAVE_EVERY_STEPS = 25          # also write the adapter to checkpoint_last/ every N steps

# Probes: the N_PROBE shortest sequences. Base NLL on their loss tokens must be below
# PROBE_MAX_NLL (the model itself sampled them at T=0.7), or the dequantization is wrong.
N_PROBE = 4
PROBE_MAX_NLL = 2.0

# Extra packages with another lora_alpha written into adapter_config.json (TASK.md §3.2:
# inference-time alpha sweep {8, 12, 16}). LORA_ALPHA itself always goes to submission.zip.
ALPHA_VARIANTS = [LORA_ALPHA, 2 * LORA_ALPHA]

# A submission that differs from the one the RFT data was sampled with stops the packaging,
# because the adapter was trained on that prompt / tool set. True = package anyway.
ALLOW_SUBMISSION_MISMATCH = False

# SMOKE = True: 2 optimizer steps on 5 sequences (4 shortest + longest), then export and
# package as submission_SMOKE.zip. Checks the full path in ~20-30 min. Do not submit it.
SMOKE = True

INSTALL_WHEELHOUSE = True      # same wheelhouse as the dev / data notebooks (Gemma 4 transformers)

assert LORA_R > 0 and LORA_ALPHA > 0 and GRAD_ACCUM >= 1
print(f'{RUN_NAME}: r={LORA_R} alpha={LORA_ALPHA} lr={LR} epochs={EPOCHS} grad_accum={GRAD_ACCUM} '
      f'targets={TARGET_MODULES} max_loss_tokens={MAX_LOSS_TOKENS} decision_weight={DECISION_WEIGHT} '
      f'first_edit_weight={FIRST_EDIT_WEIGHT} smoke={SMOKE}')


## 1. 环境

安装 wheelhouse（与 dev-eval / 数据 notebook 的 cell 1 相同），从而拿到支持 Gemma 4 的 transformers。

In [ ]:
import glob
import importlib
import logging
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

WORKING_DIR = Path('/kaggle/working')
WORKING_DIR.mkdir(parents=True, exist_ok=True)
INPUT_DIR = Path('/kaggle/input')
WHEELHOUSE_DIR = INPUT_DIR / 'datasets' / 'metric' / 'gemma-4-developer-agent-wheelhouse'

if INSTALL_WHEELHOUSE:
    assert WHEELHOUSE_DIR.is_dir(), f'wheelhouse not found at {WHEELHOUSE_DIR}: attach it or set INSTALL_WHEELHOUSE = False'
    # Same steps as cell 1 of the dev-eval / data notebooks.
    for pth_pattern in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth',
                        '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
        for pth in glob.glob(pth_pattern):
            try:
                os.unlink(pth)
            except OSError:
                pass
    # Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
    tmp_whl = Path('/tmp/wheelhouse')
    tmp_whl.mkdir(parents=True, exist_ok=True)
    for w in WHEELHOUSE_DIR.glob('*.whl'):
        if 'cutlass' in w.name.lower():
            continue
        target = tmp_whl / (w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name)
        if not target.exists():
            try:
                os.symlink(w, target)
            except OSError:
                shutil.copyfile(w, target)
    wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
    print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels], check=True)
    importlib.invalidate_caches()

for name in ('transformers', 'torch', 'urllib3', 'httpx'):
    logging.getLogger(name).setLevel(logging.ERROR)

import torch
import transformers
from transformers.models.auto.configuration_auto import CONFIG_MAPPING

N_GPUS = torch.cuda.device_count()


def gpu(i: int = 0) -> torch.device:
    return torch.device('cuda', i)


print(f'python {sys.version.split()[0]} | torch {torch.__version__} (cuda {torch.version.cuda}) | '
      f'transformers {transformers.__version__}')
for i in range(N_GPUS):
    p = torch.cuda.get_device_properties(i)
    print(f'  cuda:{i} {p.name} {p.total_memory / 1024**3:.1f} GiB')
assert 'gemma4' in CONFIG_MAPPING, f'transformers {transformers.__version__} has no Gemma 4: install the wheelhouse'
assert N_GPUS >= 1, 'no GPU: select the GPU L4x4 accelerator'
if N_GPUS != 4:
    print(f'WARNING: {N_GPUS} GPUs (expected 4 x L4). The layer split adapts, but memory may not suffice.')

## 2. Helpers

纯函数，不依赖模型或 GPU：
- int4 反量化（compressed-tensors `pack-quantized` 格式）
- `W4A16Linear` / `LoRALinear`
- 数据过滤、loss 权重、留出集与 checkpoint 挑选
- PEFT adapter 导出与 key 检查
- `agent.yaml` 修改、时间预算检查与打包（规则同 `scripts/package_submission.py`）


In [ ]:
# Pure helpers (no model, no GPU needed): int4 dequant, W4A16 / LoRA layers, data filter,
# PEFT adapter export + key check, agent.yaml edit, submission packaging.
import contextlib
import hashlib
import json
import math
import re
import zipfile
from collections import Counter
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

# ---- int4 "pack-quantized" (compressed-tensors) ----
# Packing along the input dim: value q in [-8, 7] is stored as q + 8; 8 consecutive values
# go into one int32, the first one in the lowest 4 bits (compressed_tensors pack_to_int32,
# packed_dim=1). Weight = q * scale, with one scale per output row (channel) or per
# `group_size` inputs of a row (group). Symmetric only (no zero point).


def pack_int4(q: torch.Tensor, num_bits: int = 4) -> torch.Tensor:
    """Signed ints [rows, cols] -> int32 [rows, ceil(cols * num_bits / 32)]. Self-test only."""
    pf = 32 // num_bits
    rows, cols = q.shape
    u = q.to(torch.int32) + (1 << (num_bits - 1))
    if cols % pf:
        u = F.pad(u, (0, pf - cols % pf))
    u = u.view(rows, -1, pf)
    packed = torch.zeros(u.shape[:2], dtype=torch.int32, device=q.device)
    for i in range(pf):
        packed |= u[..., i] << (num_bits * i)
    return packed


def unpack_int4(packed: torch.Tensor, cols: int, num_bits: int = 4) -> torch.Tensor:
    """int32 [rows, ceil(cols * num_bits / 32)] -> signed int32 [rows, cols]."""
    pf = 32 // num_bits
    shifts = torch.arange(pf, dtype=torch.int32, device=packed.device) * num_bits
    u = (packed.unsqueeze(-1) >> shifts) & ((1 << num_bits) - 1)   # arithmetic shift; the mask drops the sign bits
    return u.reshape(packed.shape[0], -1)[:, :cols] - (1 << (num_bits - 1))


def dequantize(packed: torch.Tensor, scale: torch.Tensor, cols: int, num_bits: int = 4,
               dtype: torch.dtype = torch.bfloat16) -> torch.Tensor:
    q = unpack_int4(packed, cols, num_bits).to(dtype)
    rows = q.shape[0]
    if scale.numel() == 1:                       # tensor strategy
        return q * scale.to(dtype).reshape(())
    s = scale.to(dtype).reshape(rows, -1)
    groups = s.shape[1]
    if groups == 1:                              # channel strategy
        return q * s
    return (q.view(rows, groups, cols // groups) * s.unsqueeze(-1)).view(rows, cols)   # group strategy


DEQUANT_FN = dequantize   # replaced by a torch.compile'd version after the self-test


class _DequantLinearFn(torch.autograd.Function):
    """y = x @ W^T with W dequantized on the fly. Saves no weight for backward: W is
    dequantized again there (W is frozen, so only grad_x is needed)."""

    @staticmethod
    def forward(ctx, x, mod):
        ctx.mod = mod
        return F.linear(x, mod.dequant().to(x.dtype))

    @staticmethod
    def backward(ctx, gy):
        return gy @ ctx.mod.dequant().to(gy.dtype), None


class W4A16Linear(nn.Module):
    def __init__(self, packed: torch.Tensor, scale: torch.Tensor, shape, num_bits: int = 4):
        super().__init__()
        self.out_features, self.in_features = int(shape[0]), int(shape[1])
        self.num_bits = num_bits
        want = (self.out_features, math.ceil(self.in_features * num_bits / 32))
        assert tuple(packed.shape) == want, f'weight_packed {tuple(packed.shape)} != {want} (only packing along the input dim is supported)'
        assert packed.dtype == torch.int32, packed.dtype
        if scale.numel() > 1:
            groups = scale.numel() // self.out_features
            assert scale.numel() == self.out_features * groups and self.in_features % groups == 0, \
                f'weight_scale {tuple(scale.shape)} does not fit weight {shape}'
        self.register_buffer('weight_packed', packed, persistent=False)
        self.register_buffer('weight_scale', scale, persistent=False)

    def dequant(self) -> torch.Tensor:
        return DEQUANT_FN(self.weight_packed, self.weight_scale, self.in_features, self.num_bits)

    def forward(self, x):
        return _DequantLinearFn.apply(x, self)

    def extra_repr(self):
        return f'in_features={self.in_features}, out_features={self.out_features}, int{self.num_bits}'


def _io_features(base: nn.Module) -> tuple[int, int, torch.device]:
    t = next(iter(list(base.buffers()) + list(base.parameters())))
    return base.in_features, base.out_features, t.device


class LoRALinear(nn.Module):
    """base(x) + B(A(dropout(x))) * alpha / r. A and B are fp32, used in the activation dtype.
    Same init as PEFT: A Kaiming-uniform, B zero (so training starts exactly at the base)."""

    def __init__(self, base: nn.Module, r: int, alpha: float, dropout: float = 0.0):
        super().__init__()
        self.base = base
        self.r, self.alpha = r, alpha
        self.scaling = alpha / r
        self.enabled = True
        in_f, out_f, dev = _io_features(base)
        self.in_features, self.out_features = in_f, out_f
        self.lora_A = nn.Parameter(torch.empty(r, in_f, device=dev, dtype=torch.float32))
        self.lora_B = nn.Parameter(torch.zeros(out_f, r, device=dev, dtype=torch.float32))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        y = self.base(x)
        if not self.enabled:
            return y
        z = F.linear(F.linear(self.dropout(x), self.lora_A.to(x.dtype)), self.lora_B.to(x.dtype))
        return y + z * self.scaling


def set_submodule(root: nn.Module, name: str, new: nn.Module) -> None:
    parent, _, attr = name.rpartition('.')
    setattr(root.get_submodule(parent) if parent else root, attr, new)


@contextlib.contextmanager
def params_on_meta():
    """Create parameters on the meta device (buffers stay real), like accelerate's
    init_empty_weights(include_buffers=False): building the 31B skeleton costs no memory."""
    old = nn.Module.register_parameter

    def register(self, name, param):
        old(self, name, param)
        if param is not None:
            p = self._parameters[name]
            self._parameters[name] = type(p)(p.to('meta'), requires_grad=p.requires_grad)

    nn.Module.register_parameter = register
    try:
        yield
    finally:
        nn.Module.register_parameter = old


# ---- data ----
def filter_rows(rows: list[dict], tool_call_id: int, end_ids: set[int], max_loss_tokens: int,
                max_seq_tokens: int, require_tool_call: bool = True) -> tuple[list[dict], Counter]:
    """Check the data notebook's invariants and keep the rows we train on."""
    kept, dropped = [], Counter()
    for r in rows:
        ids, lab = r['input_ids'], r['labels']
        assert len(ids) == len(lab), f"{r['id']}: input_ids / labels length differ"
        pos = [i for i, x in enumerate(lab) if x != -100]
        assert pos and pos[0] > 0, f"{r['id']}: no trainable token after position 0"
        assert all(lab[i] == ids[i] for i in pos), f"{r['id']}: labels differ from input_ids on loss positions"
        gen = [ids[i] for i in pos]
        if require_tool_call and tool_call_id not in gen:
            dropped['no <|tool_call> (compaction summary, not an agent turn)'] += 1
        elif len(pos) > max_loss_tokens:
            dropped[f'more than {max_loss_tokens} generated tokens'] += 1
        elif len(ids) > max_seq_tokens:
            dropped[f'sequence longer than {max_seq_tokens} tokens'] += 1
        elif end_ids and gen[-1] not in end_ids:
            dropped['generation does not end with an end-of-turn token'] += 1
        else:
            kept.append(r)
    return kept, dropped


# ---- loss weights ----
def decision_weights(labels: list[int], channel_end_id: int, tool_call_id: int, id_to_token, weight: float,
                     max_name_tokens: int = 8) -> list[float]:
    """One weight per labelled token, in order. `weight` on <channel|>, on <|tool_call> and on the
    tokens up to the '{' that opens the arguments (the tool name); 1 elsewhere."""
    out, in_name = [], 0
    for tok in labels:
        if tok == -100:
            continue
        if tok == tool_call_id:
            out.append(weight)
            in_name = max_name_tokens
        elif tok == channel_end_id:
            out.append(weight)
            in_name = 0
        elif in_name and '{' not in id_to_token(tok):
            out.append(weight)
            in_name -= 1
        else:
            out.append(1.0)
            in_name = 0
    return out


def weighted_nll(lp: torch.Tensor, weights) -> torch.Tensor:
    """Weighted mean of -log p over one sequence. All weights 1 gives -lp.mean()."""
    w = torch.as_tensor(weights, dtype=lp.dtype, device=lp.device)
    return -(lp * w).sum() / w.sum()


def adam_movement_bound(total_steps: int, warmup_steps: int, lr: float) -> float:
    """Largest distance Adam can move one parameter under the warmup + cosine schedule (about lr per step)."""
    s = 0.0
    for i in range(total_steps):
        s += (i + 1) / warmup_steps if i < warmup_steps else \
            0.5 * (1.0 + math.cos(math.pi * (i - warmup_steps) / max(1, total_steps - warmup_steps)))
    return s * lr


# ---- holdout ----
def holdout_tasks(tasks, min_tasks: int) -> list[str]:
    """Deterministic: 3 tasks (2 below 15 tasks) by hash order; none below min_tasks."""
    tasks = sorted(set(tasks))
    if len(tasks) < min_tasks:
        return []
    order = sorted(tasks, key=lambda t: hashlib.sha256(t.encode()).hexdigest())
    return sorted(order[:3 if len(tasks) >= 15 else 2])


def holdout_metrics(lps: list[torch.Tensor], rows: list[dict], channel_end_id: int, tool_call_id: int) -> dict:
    """lps[i]: log-probs of rows[i]'s labelled tokens, in order. Means over sequences; the two
    logprob means are over all tokens of that kind."""
    wnll, nll, ch, tc = [], [], [], []
    for lp, r in zip(lps, rows):
        wnll.append(weighted_nll(lp, r['loss_weights']).item())
        nll.append(-lp.mean().item())
        gen = [x for x in r['labels'] if x != -100]
        ch += [lp[i].item() for i, x in enumerate(gen) if x == channel_end_id]
        tc += [lp[i].item() for i, x in enumerate(gen) if x == tool_call_id]
    mean = lambda v: round(sum(v) / len(v), 5) if v else None
    return {'weighted_nll': mean(wnll), 'nll': mean(nll), 'channel_end_lp': mean(ch), 'tool_call_lp': mean(tc)}


class CheckpointPicker:
    """Keep the latest checkpoint whose holdout loss is within tol of the best so far."""

    def __init__(self, tol: float):
        self.tol, self.best, self.step = tol, math.inf, None

    def update(self, step: int, loss: float) -> bool:
        self.best = min(self.best, loss)
        if loss <= self.best * (1 + self.tol):
            self.step = step
            return True
        return False


# ---- PEFT adapter ----
ADAPTER_KEY_RE = re.compile(r'base_model\.model\.(.+)\.lora_([AB])\.weight')


def adapter_state_dict(loras: dict, to_ckpt_name) -> dict:
    """PEFT keys under the checkpoint's own module names, bf16, contiguous."""
    sd = {}
    for name, m in loras.items():
        key = 'base_model.model.' + to_ckpt_name(name)
        sd[key + '.lora_A.weight'] = m.lora_A.detach().to('cpu', torch.bfloat16).contiguous()
        sd[key + '.lora_B.weight'] = m.lora_B.detach().to('cpu', torch.bfloat16).contiguous()
    return sd


def adapter_config(r: int, alpha: float, target_modules, base_model: str, dropout: float = 0.0) -> dict:
    return {
        'peft_type': 'LORA',
        'task_type': 'CAUSAL_LM',
        'base_model_name_or_path': base_model,
        'r': r,
        'lora_alpha': alpha,
        'lora_dropout': dropout,
        'target_modules': sorted(target_modules),   # a list: vLLM checks each name against its LoRA modules
        'bias': 'none',
        'fan_in_fan_out': False,
        'use_rslora': False,
        'use_dora': False,
        'modules_to_save': None,
        'init_lora_weights': True,
        'inference_mode': True,
        'layers_to_transform': None,
        'layers_pattern': None,
        'rank_pattern': {},
        'alpha_pattern': {},
        'revision': None,
    }


def save_adapter(out_dir: Path, sd: dict, cfg: dict) -> None:
    from safetensors.torch import save_file
    out_dir.mkdir(parents=True, exist_ok=True)
    for p in out_dir.iterdir():   # exactly the two PEFT files, nothing else
        if p.is_file():
            p.unlink()
    save_file(sd, str(out_dir / 'adapter_model.safetensors'), metadata={'format': 'pt'})
    (out_dir / 'adapter_config.json').write_text(json.dumps(cfg, indent=2) + '\n', encoding='utf-8')


def check_adapter(adapter_dir: Path, linear_shapes: dict, r: int) -> tuple[set, list[str]]:
    """TASK.md Gate 1 key check. `linear_shapes`: checkpoint module name -> (out, in) of every
    quantized Linear in the checkpoint. Returns (module names, problems)."""
    from safetensors import safe_open
    problems, seen = [], Counter()
    files = sorted(p.name for p in adapter_dir.iterdir())
    if files != ['adapter_config.json', 'adapter_model.safetensors']:
        problems.append(f'adapter dir must hold exactly adapter_config.json + adapter_model.safetensors, has {files}')
    cfg = json.loads((adapter_dir / 'adapter_config.json').read_text(encoding='utf-8'))
    if cfg.get('r') != r or cfg.get('peft_type') != 'LORA' or cfg.get('bias') != 'none':
        problems.append(f"adapter_config: r={cfg.get('r')} peft_type={cfg.get('peft_type')} bias={cfg.get('bias')}")
    with safe_open(str(adapter_dir / 'adapter_model.safetensors'), 'pt') as f:
        for k in f.keys():
            m = ADAPTER_KEY_RE.fullmatch(k)
            if not m:
                problems.append(f'not a PEFT LoRA key: {k}')
                continue
            mod, ab = m.groups()
            seen[mod] += 1
            if mod not in linear_shapes:
                problems.append(f'{mod}: no quantized Linear of that name in the checkpoint')
                continue
            if mod.rsplit('.', 1)[-1] not in cfg.get('target_modules', []):
                problems.append(f'{mod}: not covered by target_modules {cfg.get("target_modules")}')
            out_f, in_f = linear_shapes[mod]
            sl = f.get_slice(k)
            shape, want = tuple(sl.get_shape()), ((r, in_f) if ab == 'A' else (out_f, r))
            if shape != want:
                problems.append(f'{k}: shape {shape}, expected {want}')
            if sl.get_dtype() != 'BF16':
                problems.append(f'{k}: dtype {sl.get_dtype()}, expected BF16')
    problems += [f'{mod}: lora_A / lora_B not both present' for mod, n in seen.items() if n != 2]
    if not seen:
        problems.append('adapter has no LoRA weights')
    return set(seen), problems


# ---- submission ----
def add_adapter_to_agent_yaml(text: str, adapter: str) -> str:
    """Insert `adapter: <name>` after the top-level `model:` line. Plain text edit, so
    `!include` tags survive (yaml.safe_load / dump would drop them)."""
    lines = [ln for ln in text.splitlines(keepends=True) if not re.match(r'adapter\s*:', ln)]
    out, done = [], False
    for ln in lines:
        out.append(ln)
        if not done and re.match(r'model\s*:', ln):
            out.append(('' if ln.endswith('\n') else '\n') + f'adapter: {adapter}\n')
            done = True
    if not done:
        raise ValueError('agent.yaml has no top-level `model:` line')
    return ''.join(out)


# Same rules as scripts/package_submission.py
ALLOWED_EXT = {'.yaml', '.yml', '.md', '.txt', '.py', '.json', '.safetensors'}
MAX_TOTAL_BYTES = 3 * 1024**3
SKIP_NAMES = {'.DS_Store', 'Thumbs.db'}
SKIP_DIRS = {'__pycache__', '.ipynb_checkpoints', '.git'}


N_HIDDEN_TASKS = 120
MAX_WORST_CASE_HOURS = 10.5   # of the 12 h limit; the rest is vLLM startup, sandbox setup, overshoot


def check_time_budget(src: Path) -> list[str]:
    """Worst case: every hidden task uses its whole budget. Same as scripts/package_submission.py."""
    cfg = next((c for n in ('eval_config.yaml', 'eval_config.yml') if (c := src / n).is_file()), None)
    m = re.search(r'^\s*max_time_minutes\s*:\s*([0-9.]+)', cfg.read_text(encoding='utf-8'), re.M) if cfg else None
    if not m:
        return ['eval_config.yaml has no max_time_minutes: the harness default is 60 min per task']
    minutes, problems = float(m.group(1)), []
    hours = minutes * N_HIDDEN_TASKS / 60
    if hours > MAX_WORST_CASE_HOURS:
        problems.append(f'max_time_minutes {minutes:g} x {N_HIDDEN_TASKS} tasks = {hours:g} h, over the {MAX_WORST_CASE_HOURS} h limit')
    for p in sorted((src / 'prompts').glob('*.md')):
        for said in re.findall(r'(\d+(?:\.\d+)?)\s+minutes', p.read_text(encoding='utf-8')):
            if float(said) != minutes:
                problems.append(f'{p.relative_to(src)} says {said} minutes, eval_config says {minutes:g}')
    return problems


def collect_submission(src: Path) -> tuple[list[Path], list[str]]:
    files, errors = [], []
    for p in sorted(src.rglob('*')):
        rel = p.relative_to(src)
        if any(part in SKIP_DIRS for part in rel.parts) or p.name in SKIP_NAMES or p.name.endswith(('~', '.swp', '.pyc')):
            continue
        if p.is_symlink():
            errors.append(f'symlink not allowed: {rel}')
            continue
        if p.is_dir():
            continue
        if p.suffix.lower() not in ALLOWED_EXT:
            errors.append(f'disallowed extension: {rel}')
            continue
        files.append(p)
    if not (src / 'agent.yaml').is_file():
        errors.append('agent.yaml missing at submission root')
    total = sum(p.stat().st_size for p in files)
    if total >= MAX_TOTAL_BYTES:
        errors.append(f'total size {total} bytes exceeds 3 GiB')
    return files, errors


def submission_fingerprint(root: Path, files: list[Path]) -> str:
    """Same as scripts/package_submission.py."""
    h = hashlib.sha256()
    for p in sorted(files, key=lambda p: p.relative_to(root).as_posix()):
        h.update(p.relative_to(root).as_posix().encode())
        h.update(b'\0')
        h.update(p.read_bytes())
        h.update(b'\0')
    return h.hexdigest()[:12]


def write_submission_zip(src: Path, out: Path) -> tuple[list[Path], str]:
    files, errors = collect_submission(src)
    if errors:
        raise RuntimeError('submission check failed:\n  ' + '\n  '.join(errors))
    out.unlink(missing_ok=True)
    with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as zf:
        for p in files:
            zf.write(p, p.relative_to(src).as_posix())
    return files, submission_fingerprint(src, files)


print('helpers ready')


## 3. 数据：加载 → 校验 → 过滤

- **校验**：数据 notebook 的约定——`labels` 在 loss 位置上等于 `input_ids`，且生成以 turn 结束 token 收尾。
- **过滤**（目标是简短而正确）：
  - 没有 `<|tool_call>` 的 turn（ADK 上下文压缩的 summary，不是 agent 决策）
  - 生成超过 `MAX_LOSS_TOKENS` 的 turn
  - 超过 `MAX_SEQ_TOKENS` 的序列
- **留出集**：按题留出，这些题不参加训练，只用于挑选 checkpoint。
- **loss 权重**：`loss_weights` 对应每个 loss token，`first_edit` 标记首次编辑 turn。
- **探针序列**：取最短的 `N_PROBE` 条，用于第 5 节的 base 检查和第 7 节的训练前后对比。


In [ ]:
import random
import statistics

from transformers import AutoTokenizer

MODEL_PATH = Path(MODEL_PATH)
assert (MODEL_PATH / 'config.json').is_file(), f'model not found at {MODEL_PATH}: attach gemma-4-31b-it-qat-w4a16-ct'


def find_rft_data() -> Path:
    """The directory holding train.jsonl + stats.json. A dataset upload of rft_data_<run>/
    and the data notebook's output (dir + zip of the same run) both work."""
    if RFT_DATA_DIR:
        d = Path(RFT_DATA_DIR)
        assert (d / 'train.jsonl').is_file(), f'no train.jsonl in RFT_DATA_DIR={d}'
        return d
    dirs = sorted({p.parent for p in INPUT_DIR.rglob('train.jsonl') if (p.parent / 'stats.json').is_file()})
    if not dirs:
        for z in INPUT_DIR.rglob('rft_data_*.zip'):
            dst = Path('/tmp/rft_data') / z.stem
            if not dst.exists():
                with zipfile.ZipFile(z) as zf:
                    zf.extractall(dst)
            dirs += sorted({p.parent for p in dst.rglob('train.jsonl') if (p.parent / 'stats.json').is_file()})
    by_name = {}
    for d in dirs:
        by_name.setdefault(d.name, d)
    if len(by_name) != 1:
        raise RuntimeError(f'Expected one RFT data dir (train.jsonl + stats.json) under {INPUT_DIR}, found {dirs}. '
                           'Attach it, or set RFT_DATA_DIR.')
    return next(iter(by_name.values()))


DATA_DIR = find_rft_data()
DATA_STATS = json.loads((DATA_DIR / 'stats.json').read_text(encoding='utf-8'))
ALL_ROWS = [json.loads(l) for l in (DATA_DIR / 'train.jsonl').read_text(encoding='utf-8').splitlines() if l.strip()]
print(f'RFT data: {DATA_DIR} (run {DATA_STATS.get("run_name")}, model {DATA_STATS.get("model")}, '
      f'submission {DATA_STATS.get("submission_fingerprint")}) -> {len(ALL_ROWS)} sequences')
assert DATA_STATS.get('model') in (None, 'gemma-4-31b-it-qat-w4a16-ct'), f'data sampled from {DATA_STATS.get("model")}'

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH))
TOOL_CALL_ID = tokenizer.convert_tokens_to_ids('<|tool_call>')
assert TOOL_CALL_ID is not None and TOOL_CALL_ID != tokenizer.unk_token_id, '<|tool_call> is not a single token'
CHANNEL_END_ID = tokenizer.convert_tokens_to_ids('<channel|>')
assert CHANNEL_END_ID is not None and CHANNEL_END_ID != tokenizer.unk_token_id, '<channel|> is not a single token'
END_IDS = set(DATA_STATS.get('end_of_turn_ids') or [])
print('tool call id:', TOOL_CALL_ID, '| end-of-turn ids:', {i: tokenizer.convert_ids_to_tokens(i) for i in sorted(END_IDS)})

TRAIN, DROPPED = filter_rows(ALL_ROWS, TOOL_CALL_ID, END_IDS, MAX_LOSS_TOKENS, MAX_SEQ_TOKENS, REQUIRE_TOOL_CALL)
for row in TRAIN:
    row['n_loss_tokens'] = sum(x != -100 for x in row['labels'])
    row['n_tokens'] = len(row['input_ids'])
    row['loss_weights'] = decision_weights(row['labels'], CHANNEL_END_ID, TOOL_CALL_ID,
                                           tokenizer.convert_ids_to_tokens, DECISION_WEIGHT)
print(f'kept {len(TRAIN)} / {len(ALL_ROWS)} sequences; dropped: {dict(DROPPED)}')
assert TRAIN, 'no training sequences left'
if 'first_edit' not in TRAIN[0]:
    print('WARNING: this data has no first_edit field (older data notebook): FIRST_EDIT_WEIGHT has no effect.')

# Holdout, by task: these tasks never reach the optimizer.
HOLDOUT_TASKS = holdout_tasks((r['task_id'] for r in TRAIN), HOLDOUT_MIN_TASKS)
HOLDOUT = sorted((r for r in TRAIN if r['task_id'] in HOLDOUT_TASKS), key=lambda r: r['n_tokens'])[:HOLDOUT_MAX_SEQS]
TRAIN = [r for r in TRAIN if r['task_id'] not in HOLDOUT_TASKS]
if SMOKE:
    HOLDOUT = HOLDOUT[:2]
print(f'holdout: {len(HOLDOUT)} sequences of tasks {HOLDOUT_TASKS}; training on {len(TRAIN)} sequences')
if not HOLDOUT:
    print(f'WARNING: fewer than {HOLDOUT_MIN_TASKS} tasks, so no holdout: the last checkpoint gets exported.')
assert TRAIN, 'no training sequences left after the holdout'

# Probes: the shortest sequences (cheap); the base-NLL check and the after-training delta use them.
PROBE_ROWS = sorted(TRAIN, key=lambda r: r['n_tokens'])[:N_PROBE]
LONGEST_ROW = max(TRAIN, key=lambda r: r['n_tokens'])
if SMOKE:
    TRAIN = PROBE_ROWS + [LONGEST_ROW]
    GRAD_ACCUM = math.ceil(len(TRAIN) / 2)    # 2 optimizer steps
    EPOCHS = 1
    print(f'SMOKE: {len(TRAIN)} sequences, grad_accum={GRAD_ACCUM}, epochs={EPOCHS}')

_tool_names = Counter()
for row in TRAIN:
    gen = tokenizer.decode([t for t in row['labels'] if t != -100])
    _tool_names.update(re.findall(r'<\|tool_call>call:(\w+)', gen))
_ntok = [r['n_tokens'] for r in TRAIN]
_nloss = [r['n_loss_tokens'] for r in TRAIN]
DATA_SUMMARY = {
    'data_dir': str(DATA_DIR),
    'data_run': DATA_STATS.get('run_name'),
    'n_rows_in': len(ALL_ROWS),
    'n_rows_train': len(TRAIN),
    'dropped': dict(DROPPED),
    'n_tasks': len({r['task_id'] for r in TRAIN}),
    'holdout_tasks': HOLDOUT_TASKS,
    'n_holdout_rows': len(HOLDOUT),
    'n_first_edit_rows': sum(bool(r.get('first_edit')) for r in TRAIN),
    'mean_weighted_token_share': round(statistics.mean(sum(x != 1.0 for x in r['loss_weights']) / len(r['loss_weights']) for r in TRAIN), 4),
    'tokens': sum(_ntok),
    'loss_tokens': sum(_nloss),
    'seq_tokens_median_max': [statistics.median(_ntok), max(_ntok)],
    'loss_tokens_median_max': [statistics.median(_nloss), max(_nloss)],
    'tool_calls': dict(_tool_names.most_common()),
}
print(json.dumps(DATA_SUMMARY, indent=1))
_ex = PROBE_ROWS[0]
print('\n--- example loss turn (shortest sequence):')
print(tokenizer.decode([t for t in _ex['labels'] if t != -100])[:1500])


## 4. 反量化与 LoRA 自检

- pack/unpack 往返，并与直接计算对照。
- 装了 `compressed_tensors` 时，与它的 `pack_to_int32` 对照位布局。
- 自定义 autograd 的 `grad_x` 与 `F.linear` 对照。
- B=0 时 LoRA 输出等于 base。
- 可选：`torch.compile` 反量化。

In [ ]:
# 1. pack -> unpack round trip, and the group / channel dequant against a direct computation
torch.manual_seed(0)
_dev = gpu(0)
for rows, cols, groups in ((48, 256, 2), (40, 136, 1), (16, 64, 64 // 32)):
    q = torch.randint(-8, 8, (rows, cols), dtype=torch.int32, device=_dev)
    packed = pack_int4(q)
    assert torch.equal(unpack_int4(packed, cols), q), 'int4 pack/unpack round trip failed'
    scale = torch.rand(rows, groups, device=_dev, dtype=torch.bfloat16) + 0.01
    ref = (q.to(torch.float32) * scale.float().repeat_interleave(cols // groups, dim=1)).to(torch.bfloat16)
    assert torch.equal(dequantize(packed, scale, cols), ref), 'dequantize mismatch'

# 2. same bit layout as compressed-tensors' own packer, when the library is installed
_ct_pack = None
for _mod in ('compressed_tensors.compressors.quantized_compressors.pack_quantized',
             'compressed_tensors.compressors.quantized_compressors.helpers',
             'compressed_tensors.compressors.pack_quantized'):
    try:
        _ct_pack = getattr(importlib.import_module(_mod), 'pack_to_int32')
        break
    except (ImportError, AttributeError):
        continue
if _ct_pack is not None:
    q = torch.randint(-8, 8, (32, 128), dtype=torch.int8)
    try:
        _theirs = torch.as_tensor(_ct_pack(q, 4)).to(torch.int32)
    except Exception as e:  # noqa: BLE001 -- signature differs across versions
        _theirs = None
        print(f'could not call {_ct_pack.__module__}.pack_to_int32 ({type(e).__name__}: {e}); '
              'layout is checked by the base-NLL probe in section 5 instead')
    if _theirs is not None:
        assert torch.equal(_theirs, pack_int4(q)), 'bit layout differs from compressed_tensors.pack_to_int32'
        print(f'pack layout matches compressed_tensors ({_ct_pack.__module__})')
else:
    print('compressed_tensors not installed: layout is checked by the base-NLL probe in section 5 instead')

# 3. custom autograd: grad_x equals the F.linear reference; the weight gets no gradient
q = torch.randint(-8, 8, (24, 64), dtype=torch.int32, device=_dev)
lin = W4A16Linear(pack_int4(q), torch.rand(24, 2, device=_dev, dtype=torch.bfloat16) + 0.01, (24, 64))
x = torch.randn(3, 5, 64, device=_dev, dtype=torch.bfloat16, requires_grad=True)
x2 = x.detach().clone().requires_grad_(True)
lin(x).float().square().sum().backward()
F.linear(x2, lin.dequant()).float().square().sum().backward()
assert torch.allclose(x.grad.float(), x2.grad.float(), atol=1e-2, rtol=1e-2), 'grad_x mismatch'

# 4. LoRA starts at the base (B = 0) and only A / B get gradients
lora = LoRALinear(lin, r=4, alpha=4, dropout=0.0)
y0 = lin(x.detach())
assert torch.equal(lora(x.detach()), y0), 'LoRA with B=0 must equal the base'
lora.lora_B.data.normal_()
lora(x.detach()).float().sum().backward()
assert lora.lora_A.grad is not None and lora.lora_B.grad is not None
assert not any(b.requires_grad for b in lin.buffers())
del q, lin, x, x2, lora, y0

# 5. optional: fused dequant via torch.compile (falls back to eager on any error)
if COMPILE_DEQUANT:
    try:
        # one graph per weight shape (q/k/v/o/gate/up/down, local and global layers)
        for _knob in ('cache_size_limit', 'recompile_limit'):
            if hasattr(torch._dynamo.config, _knob):
                setattr(torch._dynamo.config, _knob, max(64, getattr(torch._dynamo.config, _knob)))
        _compiled = torch.compile(dequantize, dynamic=False)
        q = torch.randint(-8, 8, (256, 512), dtype=torch.int32, device=_dev)
        p, s = pack_int4(q), torch.rand(256, 4, device=_dev, dtype=torch.bfloat16) + 0.01
        assert torch.equal(_compiled(p, s, 512), dequantize(p, s, 512))
        DEQUANT_FN = _compiled
        print('dequant: torch.compile')
    except Exception as e:  # noqa: BLE001
        DEQUANT_FN = dequantize
        print(f'dequant: eager (torch.compile failed: {type(e).__name__}: {str(e)[:200]})')
print('self-test passed')

## 5. 建模型、加载 w4a16 权重、分卡、挂 LoRA

- 只建文本模型（不建视觉塔），参数先放 meta。
- 在 safetensors index 里找到语言模型前缀 `P`（如 `model.language_model.`）：本地 `model.X` 对应 checkpoint 的 `P + X`。
- int4 Linear 换成 `W4A16Linear`，其余参数以 bf16 加载。checkpoint 里存了值的 buffer（Gemma 4 每层的 `layer_scalar`）也一并加载。缺任何参数、或 checkpoint 里有语言模型的 key 没被加载，都直接报错。
- decoder 层按“权重 + 激活”连续地分到 4 张卡。GPU 0 还放 embedding 和 loss 位置的 logits，所以分到的层更少。
- 每层挂 pre-hook 搬输入；mask / RoPE 每步每卡只拷一次。
- 在语言模型层的 `q/k/v/o_proj` 上挂 LoRA，开 non-reentrant gradient checkpointing。

In [ ]:
import gc
from collections import defaultdict

from safetensors import safe_open
from transformers import AutoConfig, AutoModelForCausalLM

# ---- quantization config of the checkpoint ----
cfg = AutoConfig.from_pretrained(str(MODEL_PATH))
QCFG = cfg.to_dict().get('quantization_config') or {}
assert QCFG.get('quant_method') == 'compressed-tensors', f"expected a compressed-tensors checkpoint, got {QCFG.get('quant_method')}"
assert QCFG.get('format') in ('pack-quantized', None), f"unsupported compressed-tensors format {QCFG.get('format')}"
_wargs = [g.get('weights') or {} for g in (QCFG.get('config_groups') or {}).values()]
assert _wargs, 'quantization_config has no config_groups'
assert {a.get('num_bits') for a in _wargs} == {4}, f"expected int4 weights: {_wargs}"
assert all(a.get('type', 'int') == 'int' and a.get('symmetric', True) for a in _wargs), \
    f'only symmetric int weights are supported: {_wargs}'
assert not any(a.get('actorder') for a in _wargs), f'act-order (g_idx) weights are not supported: {_wargs}'
NUM_BITS = 4
print('quantization:', [{k: a.get(k) for k in ('num_bits', 'strategy', 'group_size', 'symmetric')} for a in _wargs])

# ---- text-only model skeleton (no vision tower), parameters on meta ----
text_cfg = getattr(cfg, 'text_config', None) or cfg
for _c in {id(cfg): cfg, id(text_cfg): text_cfg}.values():
    if hasattr(_c, 'quantization_config'):
        try:
            delattr(_c, 'quantization_config')
        except AttributeError:
            _c.quantization_config = None
text_cfg.use_cache = False


def _build(config):
    for kw in ({'torch_dtype': torch.bfloat16}, {'dtype': torch.bfloat16}):
        try:
            return AutoModelForCausalLM.from_config(config, attn_implementation=ATTN_IMPL, **kw)
        except TypeError:
            continue
        except ValueError:   # text config not registered for AutoModelForCausalLM
            break
    mod = importlib.import_module(f'transformers.models.{cfg.model_type}.modeling_{cfg.model_type}')
    cls = next(getattr(mod, n) for n in dir(mod) if n.endswith('ForCausalLM'))
    config._attn_implementation = ATTN_IMPL
    return cls(config)


_old_dtype = torch.get_default_dtype()
torch.set_default_dtype(torch.bfloat16)
try:
    with params_on_meta():
        model = _build(text_cfg)
finally:
    torch.set_default_dtype(_old_dtype)
DECODER = model.model
LAYERS = DECODER.layers
N_LAYERS = len(LAYERS)
print(f'{type(model).__name__}: {N_LAYERS} layers, hidden {text_cfg.hidden_size}, '
      f'vocab {text_cfg.vocab_size}, attn={model.config._attn_implementation}')

# ---- checkpoint index and the name map ----
_idx = MODEL_PATH / 'model.safetensors.index.json'
if _idx.is_file():
    WEIGHT_MAP = json.loads(_idx.read_text())['weight_map']
else:
    with safe_open(str(MODEL_PATH / 'model.safetensors'), 'pt') as f:
        WEIGHT_MAP = {k: 'model.safetensors' for k in f.keys()}
_anchor = 'layers.0.self_attn.q_proj.weight_packed'
_hits = [k for k in WEIGHT_MAP if k.endswith(_anchor) and not re.search(r'vision|audio|tower', k)]
assert len(_hits) == 1, f'cannot locate the language model in the checkpoint: {_hits}'
CKPT_PREFIX = _hits[0][:-len(_anchor)]          # e.g. 'model.language_model.'
print('checkpoint language-model prefix:', repr(CKPT_PREFIX))


def to_ckpt(local: str) -> str:
    """Local module / parameter name -> checkpoint name ('model.X' -> CKPT_PREFIX + 'X')."""
    return CKPT_PREFIX + local[len('model.'):] if local.startswith('model.') else local


DTYPE_BYTES = {'BF16': 2, 'F16': 2, 'F32': 4, 'F64': 8, 'I64': 8, 'I32': 4, 'I16': 2, 'I8': 1, 'U8': 1, 'BOOL': 1}
TENSOR_BYTES, TENSOR_SHAPES = {}, {}
for _shard in sorted(set(WEIGHT_MAP.values())):
    with safe_open(str(MODEL_PATH / _shard), 'pt') as f:
        for k in f.keys():
            sl = f.get_slice(k)
            TENSOR_SHAPES[k] = tuple(sl.get_shape())
            TENSOR_BYTES[k] = math.prod(TENSOR_SHAPES[k]) * DTYPE_BYTES.get(sl.get_dtype(), 4)

# ---- what to load: quantized Linears, plain parameters ----
QUANT_LINEARS, PLAIN_PARAMS = [], []
for name, mod in model.named_modules():
    if isinstance(mod, nn.Linear) and to_ckpt(name) + '.weight_packed' in WEIGHT_MAP:
        QUANT_LINEARS.append(name)
        for extra in ('weight_zero_point', 'weight_g_idx'):
            assert to_ckpt(name) + '.' + extra not in WEIGHT_MAP, f'{name}: {extra} is not supported'
_quant = set(QUANT_LINEARS)
TIED_LM_HEAD = 'lm_head.weight' not in WEIGHT_MAP
_missing = []
for pname, _ in model.named_parameters(remove_duplicate=False):
    owner = pname.rpartition('.')[0]
    if owner in _quant or (TIED_LM_HEAD and pname == 'lm_head.weight'):
        continue
    (PLAIN_PARAMS if to_ckpt(pname) in WEIGHT_MAP else _missing).append(pname)
assert not _missing, f'parameters missing from the checkpoint: {_missing[:20]}'
assert QUANT_LINEARS, 'no quantized Linear found under the language model'
# persistent buffers whose values live in the checkpoint (Gemma 4: layers.N.layer_scalar)
CKPT_BUFFERS = [b for b, _ in model.named_buffers() if to_ckpt(b) in WEIGHT_MAP]
print('buffers loaded from the checkpoint:', len(CKPT_BUFFERS), CKPT_BUFFERS[:3])

# ---- layer -> GPU: contiguous groups, balanced by weight bytes + checkpointed activations ----
def _layer_of(local: str):
    m = re.match(r'model\.layers\.(\d+)\.', local)
    return int(m.group(1)) if m else None


_lbytes = [0] * N_LAYERS
_other_bytes = 0
for local in [n + s for n in QUANT_LINEARS for s in ('.weight_packed', '.weight_scale')] + PLAIN_PARAMS:
    i, b = _layer_of(local), TENSOR_BYTES[to_ckpt(local)]
    if i is None:
        _other_bytes += b
    else:
        _lbytes[i] += b
_act = MAX_SEQ_TOKENS * text_cfg.hidden_size * 2          # one bf16 hidden state kept per layer
_cost = [b + _act for b in _lbytes]
if LAYER_SPLIT:
    assert sum(LAYER_SPLIT) == N_LAYERS and len(LAYER_SPLIT) <= N_GPUS, LAYER_SPLIT
    LAYER_DEV = [d for d, n in enumerate(LAYER_SPLIT) for _ in range(n)]
else:
    _reserve0 = _other_bytes + GPU0_RESERVE_GB * 1024**3
    _target = (sum(_cost) + _reserve0) / N_GPUS
    LAYER_DEV, _acc = [], _reserve0
    for c in _cost:
        LAYER_DEV.append(min(N_GPUS - 1, int((_acc + c / 2) // _target)))
        _acc += c
print('layers per GPU:', dict(Counter(LAYER_DEV)), '| weights per GPU (GiB):',
      {d: round(sum(b for b, ld in zip(_lbytes, LAYER_DEV) if ld == d) / 1024**3 + (_other_bytes / 1024**3 if d == 0 else 0), 2)
       for d in range(N_GPUS)})


def device_of(local: str) -> torch.device:
    i = _layer_of(local)
    return gpu(LAYER_DEV[i] if i is not None else 0)


# ---- load tensors shard by shard, straight to their GPU ----
_wanted = {}                                     # ckpt key -> (device, cast to bf16?)
for n in QUANT_LINEARS:
    dev = device_of(n)
    _wanted[to_ckpt(n) + '.weight_packed'] = (dev, False)
    _wanted[to_ckpt(n) + '.weight_scale'] = (dev, False)
    if to_ckpt(n) + '.weight_shape' in WEIGHT_MAP:
        _wanted[to_ckpt(n) + '.weight_shape'] = (torch.device('cpu'), False)
for pname in PLAIN_PARAMS + CKPT_BUFFERS:
    _wanted[to_ckpt(pname)] = (device_of(pname), True)
_by_shard = defaultdict(list)
for k in _wanted:
    _by_shard[WEIGHT_MAP[k]].append(k)
_store, _t0 = {}, time.time()
for si, (_shard, keys) in enumerate(sorted(_by_shard.items())):
    with safe_open(str(MODEL_PATH / _shard), 'pt', device='cpu') as f:
        for k in keys:
            t = f.get_tensor(k)
            dev, cast = _wanted[k]
            if cast and t.is_floating_point():
                t = t.to(torch.bfloat16)
            _store[k] = t.to(dev)
    print(f'  shard {si + 1}/{len(_by_shard)} {_shard} ({time.time() - _t0:.0f}s)')

for n in QUANT_LINEARS:
    ck = to_ckpt(n)
    shape = _store.pop(ck + '.weight_shape').tolist() if ck + '.weight_shape' in _store else \
        [TENSOR_SHAPES[ck + '.weight_packed'][0], model.get_submodule(n).in_features]
    set_submodule(model, n, W4A16Linear(_store.pop(ck + '.weight_packed'), _store.pop(ck + '.weight_scale'), shape, NUM_BITS))
for pname in PLAIN_PARAMS:
    owner, _, attr = pname.rpartition('.')
    model.get_submodule(owner)._parameters[attr] = nn.Parameter(_store.pop(to_ckpt(pname)), requires_grad=False)
for bname in CKPT_BUFFERS:
    owner, _, attr = bname.rpartition('.')
    old = model.get_submodule(owner)._buffers[attr]
    t = _store.pop(to_ckpt(bname))
    assert t.shape == old.shape, f'{bname}: checkpoint {tuple(t.shape)} != model {tuple(old.shape)}'
    model.get_submodule(owner)._buffers[attr] = t.to(old.dtype)
assert not _store
if TIED_LM_HEAD:
    model.lm_head.weight = DECODER.embed_tokens.weight
LM_HEAD_W = model.lm_head.weight
SOFTCAP = getattr(text_cfg, 'final_logit_softcapping', None)

# Buffers were built on the CPU (only parameters went to meta): move each to its module's GPU.
# If this transformers version built some on meta anyway (e.g. rotary inv_freq), rebuild
# that parameter-free module from the config on its GPU.
def _rebuild(owner: str, dev: torch.device) -> None:
    m = model.get_submodule(owner)
    assert not list(m.parameters(recurse=False)), f'{owner}: meta buffers in a module with parameters'
    for args, kw in (((), {'config': text_cfg, 'device': dev}), ((text_cfg, dev), {}), ((text_cfg,), {})):
        try:
            new = type(m)(*args, **kw)
            break
        except TypeError:
            continue
    else:
        raise RuntimeError(f'cannot rebuild {owner} ({type(m).__name__}) whose buffers are on meta')
    set_submodule(model, owner, new.to(dev))
    print(f'  rebuilt {owner} ({type(m).__name__}) on {dev}')


for bname, buf in list(model.named_buffers()):
    owner, _, attr = bname.rpartition('.')
    if buf.is_meta:
        if attr == 'embed_scale':   # Gemma scales embeddings by sqrt(hidden_size)
            model.get_submodule(owner)._buffers[attr] = torch.tensor(text_cfg.hidden_size ** 0.5, dtype=torch.bfloat16, device=device_of(bname))
        elif any(b.is_meta for b in model.get_submodule(owner).buffers(recurse=False)):
            _rebuild(owner, device_of(bname))
        continue
    model.get_submodule(owner)._buffers[attr] = buf.to(device_of(bname))
_meta = [n for n, b in model.named_buffers() if b.is_meta]
assert not _meta, f'buffers still on meta: {_meta}'
for pname, p in model.named_parameters():
    assert p.device.type == gpu(0).type, f'{pname} is on {p.device}'
    p.requires_grad_(False)

_used = ({to_ckpt(p) for p in PLAIN_PARAMS} | {to_ckpt(b) for b in CKPT_BUFFERS}
         | {to_ckpt(n) + s for n in QUANT_LINEARS for s in ('.weight_packed', '.weight_scale', '.weight_shape')})
_unused = [k for k in WEIGHT_MAP if k.startswith(CKPT_PREFIX) and k not in _used]
assert not _unused, f'checkpoint language-model keys not loaded: {_unused[:20]}'
print(f'loaded {len(QUANT_LINEARS)} int4 Linears + {len(PLAIN_PARAMS)} plain tensors in {time.time() - _t0:.0f}s; '
      f'tied lm_head={TIED_LM_HEAD}; final_logit_softcapping={SOFTCAP}; '
      f'unused language-model keys: {len(_unused)} {_unused[:8]}; other (vision etc.) keys skipped: '
      f'{sum(not k.startswith(CKPT_PREFIX) for k in WEIGHT_MAP)}')
gc.collect()

# ---- cross-GPU input moves: pre-hook on every layer / top-level module ----
HOOK_CACHE = {}          # (id(src), device) -> (src, copy); cleared before each sequence
_CACHE_MIN = 1 << 16


def _move(obj, dev):
    if isinstance(obj, torch.Tensor):
        if obj.device == dev:
            return obj
        if obj.requires_grad or obj.numel() < _CACHE_MIN:
            return obj.to(dev)
        key = (id(obj), dev)
        hit = HOOK_CACHE.get(key)
        if hit is None or hit[0] is not obj:
            hit = HOOK_CACHE[key] = (obj, obj.to(dev))   # keeping `obj` alive pins its id
        return hit[1]
    if isinstance(obj, tuple):
        return tuple(_move(o, dev) for o in obj)
    if isinstance(obj, list):
        return [_move(o, dev) for o in obj]
    if isinstance(obj, dict):
        return {k: _move(v, dev) for k, v in obj.items()}
    return obj


def _pre_hook(dev):
    def hook(module, args, kwargs):
        return _move(args, dev), _move(kwargs, dev)
    return hook


for i, layer in enumerate(LAYERS):
    layer.register_forward_pre_hook(_pre_hook(gpu(LAYER_DEV[i])), with_kwargs=True)
for cname, child in DECODER.named_children():
    if cname != 'layers':
        child.register_forward_pre_hook(_pre_hook(gpu(0)), with_kwargs=True)

# ---- LoRA on the attention projections of the language-model layers ----
# checkpoint name -> (out, in) of every int4 Linear, for the adapter key check (section 9)
CKPT_LINEAR_SHAPES = {to_ckpt(n): (m.out_features, m.in_features) for n, m in model.named_modules()
                      if isinstance(m, W4A16Linear)}
LORAS = {}
for name, mod in list(model.named_modules()):
    if isinstance(mod, W4A16Linear) and name.rsplit('.', 1)[-1] in TARGET_MODULES and _layer_of(name) is not None:
        LORAS[name] = LoRALinear(mod, LORA_R, LORA_ALPHA, LORA_DROPOUT)
        set_submodule(model, name, LORAS[name])
_per_target = Counter(n.rsplit('.', 1)[-1] for n in LORAS)
LORA_TARGETS = sorted(_per_target)
print('LoRA modules:', dict(_per_target), f'of {N_LAYERS} layers;',
      f'{sum(p.numel() for m in LORAS.values() for p in (m.lora_A, m.lora_B)) / 1e6:.1f}M trainable params')
for t in TARGET_MODULES:
    if _per_target[t] < N_LAYERS:
        _without = [i for i in range(N_LAYERS) if f'model.layers.{i}.self_attn.{t}' not in LORAS]
        print(f'  note: {t} missing in {len(_without)} layers {_without[:12]} (e.g. layers whose K and V share a projection)')
assert LORAS, f'no {TARGET_MODULES} found'

model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
model.train()
for d in range(N_GPUS):
    print(f'  cuda:{d} allocated {torch.cuda.memory_allocated(d) / 1024**3:.2f} GiB')

## 6. Base 检查 + 显存探测

- **Base NLL**：探针序列的 loss token 是模型自己在 T=0.7 下采出来的，NLL 应远小于 `PROBE_MAX_NLL`。若接近 10，说明反量化或名字映射错了。
- **显存探测**：在最长序列上跑一次 forward + backward，打印每卡峰值显存和粗略的训练时长估计。

In [ ]:
def seq_logprobs(row: dict) -> torch.Tensor:
    """Log-probs of the trained tokens of one sequence. Logits only at the loss positions:
    full 16k x 262k logits would need ~17 GB."""
    HOOK_CACHE.clear()
    ids = torch.tensor(row['input_ids'], device=gpu(0)).unsqueeze(0)
    lab = torch.tensor(row['labels'], device=gpu(0))
    pos = (lab[1:] != -100).nonzero().squeeze(1)       # position p predicts token p + 1
    tgt = lab[1:].index_select(0, pos)
    hidden = DECODER(input_ids=ids, use_cache=False).last_hidden_state[0]
    h = hidden.index_select(0, pos.to(hidden.device)).to(LM_HEAD_W.device)
    logits = F.linear(h, LM_HEAD_W).float()
    if SOFTCAP:
        logits = torch.tanh(logits / SOFTCAP) * SOFTCAP
    return -F.cross_entropy(logits, tgt.to(logits.device), reduction='none')


@torch.no_grad()
def probe(rows: list[dict]) -> list[torch.Tensor]:
    was_training = model.training
    model.eval()
    try:
        return [seq_logprobs(r).cpu() for r in rows]
    finally:
        model.train(was_training)


def holdout_eval() -> dict | None:
    return holdout_metrics(probe(HOLDOUT), HOLDOUT, CHANNEL_END_ID, TOOL_CALL_ID) if HOLDOUT else None


def set_lora(enabled: bool) -> None:
    for m in LORAS.values():
        m.enabled = enabled


def peak_gib() -> dict:
    return {d: round(torch.cuda.max_memory_allocated(d) / 1024**3, 1) for d in range(N_GPUS)}


# 1. base check: the model sampled these tokens itself (T=0.7), so their NLL must be low.
t0 = time.time()
BASE_LP = probe(PROBE_ROWS)
BASE_NLL = [round(-lp.mean().item(), 4) for lp in BASE_LP]
print(f'base NLL on the loss tokens of {len(PROBE_ROWS)} probe sequences: {BASE_NLL} ({time.time() - t0:.0f}s)')
assert statistics.mean(BASE_NLL) < PROBE_MAX_NLL, (
    f'base NLL {BASE_NLL} >= {PROBE_MAX_NLL}: the int4 weights are not dequantized as vLLM does '
    '(check packing / scale layout) or the checkpoint name map is wrong.')

# LoRA is attached with B = 0, so this is the base model on the holdout.
t0 = time.time()
HOLDOUT_BASE = holdout_eval()
print(f'holdout base: {HOLDOUT_BASE} ({time.time() - t0:.0f}s per evaluation)')

# 2. memory: forward + backward on the longest sequence (the worst case of the run)
for d in range(N_GPUS):
    torch.cuda.reset_peak_memory_stats(d)
t0 = time.time()
(-seq_logprobs(LONGEST_ROW).mean()).backward()
torch.cuda.synchronize()
_dt = time.time() - t0
for m in LORAS.values():
    m.lora_A.grad = m.lora_B.grad = None
SEC_PER_TOKEN = _dt / LONGEST_ROW['n_tokens']
_est_h = SEC_PER_TOKEN * sum(r['n_tokens'] for r in TRAIN) * EPOCHS / 3600
print(f"longest sequence ({LONGEST_ROW['n_tokens']} tokens): fwd+bwd {_dt:.0f}s, peak GiB per GPU {peak_gib()}")
print(f'rough training estimate: {_est_h:.2f} h for {sum(r["n_tokens"] for r in TRAIN) * EPOCHS / 1e6:.2f}M tokens '
      '(the first step includes compile / warm-up; the log below gives the real rate)')


## 7. 训练

- 每个序列先在自己的 loss token 上按 `loss_weights` 取加权平均，再在 batch 内按序列权重取平均（首次编辑 turn 为 `FIRST_EDIT_WEIGHT`）。
- AdamW + cosine 调度，warmup 占 `WARMUP_RATIO`，`clip_grad_norm=MAX_GRAD_NORM`。
- 每 `SAVE_EVERY_STEPS` 步写一次 `checkpoint_last/`。
- 每 `SAVE_EVERY_STEPS` 步、以及训练结束时评估留出集，最终导出 `CheckpointPicker` 选中的权重。
- 超过 `MAX_TRAIN_HOURS` 就停止训练，照常导出。


In [ ]:
PARAMS = [p for m in LORAS.values() for p in (m.lora_A, m.lora_B)]
for p in PARAMS:
    p.requires_grad_(True)
optimizer = torch.optim.AdamW(PARAMS, lr=LR, betas=(0.9, 0.999), weight_decay=WEIGHT_DECAY)

_rng = random.Random(SEED)
ORDER = []
for _ in range(EPOCHS):
    _o = list(range(len(TRAIN)))
    _rng.shuffle(_o)
    ORDER += _o
BATCHES = [ORDER[i:i + GRAD_ACCUM] for i in range(0, len(ORDER), GRAD_ACCUM)]
TOTAL_STEPS = len(BATCHES)
WARMUP_STEPS = max(1, round(WARMUP_RATIO * TOTAL_STEPS))


def lr_factor(step: int) -> float:
    """Linear warmup, then cosine to 0 (HF get_cosine_schedule_with_warmup)."""
    if step < WARMUP_STEPS:
        return (step + 1) / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / max(1, TOTAL_STEPS - WARMUP_STEPS)
    return 0.5 * (1.0 + math.cos(math.pi * progress))


ADAPTER_BASE_NAME = MODEL_PATH.parent.name if MODEL_PATH.name.isdigit() else MODEL_PATH.name
CKPT_DIR = WORKING_DIR / 'checkpoint_last'
LOG_PATH = WORKING_DIR / 'train_log.jsonl'
LOG_PATH.unlink(missing_ok=True)


def write_adapter(out_dir: Path, alpha: float = LORA_ALPHA) -> None:
    save_adapter(out_dir, adapter_state_dict(LORAS, to_ckpt),
                 adapter_config(LORA_R, alpha, LORA_TARGETS, ADAPTER_BASE_NAME, LORA_DROPOUT))


def seq_weight(row: dict) -> float:
    return FIRST_EDIT_WEIGHT if row.get('first_edit') else 1.0


def snapshot_lora() -> dict:
    return {n: (m.lora_A.detach().cpu().clone(), m.lora_B.detach().cpu().clone()) for n, m in LORAS.items()}


def restore_lora(snap: dict) -> None:
    for n, (a, b) in snap.items():
        LORAS[n].lora_A.data.copy_(a.to(LORAS[n].lora_A.device))
        LORAS[n].lora_B.data.copy_(b.to(LORAS[n].lora_B.device))


PICKER = CheckpointPicker(HOLDOUT_TOL)
SELECTED_LORA, HOLDOUT_HISTORY = None, []


def eval_and_pick(step: int) -> None:
    global SELECTED_LORA
    m = holdout_eval()
    if m is None:
        return
    picked = PICKER.update(step, m['weighted_nll'])
    if picked:
        SELECTED_LORA = snapshot_lora()
    HOLDOUT_HISTORY.append({'step': step, 'picked': picked, **m})
    print(f'  holdout @ {step}: {m}' + ('  <- picked' if picked else ''))


HISTORY, STOP_REASON = [], None
_tok_total = sum(TRAIN[j]['n_tokens'] for b in BATCHES for j in b)
_tok_done, _t_train = 0, time.time()
print(f'{len(TRAIN)} sequences x {EPOCHS} epoch -> {TOTAL_STEPS} optimizer steps of {GRAD_ACCUM}, '
      f'warmup {WARMUP_STEPS}, lr {LR}')
model.train()
for step, batch in enumerate(BATCHES):
    _hours = (time.time() - NOTEBOOK_START) / 3600
    if _hours > MAX_TRAIN_HOURS:
        STOP_REASON = f'time guard: {_hours:.2f} h > MAX_TRAIN_HOURS={MAX_TRAIN_HOURS} after {step} steps'
        print('STOP:', STOP_REASON)
        break
    for g in optimizer.param_groups:
        g['lr'] = LR * lr_factor(step)
    t0, losses, ntok = time.time(), [], 0
    _batch_w = sum(seq_weight(TRAIN[j]) for j in batch)
    for j in batch:
        row = TRAIN[j]
        try:
            loss = weighted_nll(seq_logprobs(row), row['loss_weights'])
            (loss * seq_weight(row) / _batch_w).backward()
        except torch.cuda.OutOfMemoryError:
            print(f"OOM on {row['id']} ({row['n_tokens']} tokens), peak GiB {peak_gib()}. "
                  'Lower MAX_SEQ_TOKENS, set LAYER_SPLIT, or raise GPU0_RESERVE_GB if cuda:0 is the one full.')
            raise
        losses.append(loss.item())
        ntok += row['n_tokens']
    grad_norm = torch.nn.utils.clip_grad_norm_(PARAMS, MAX_GRAD_NORM).item()
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)
    _tok_done += ntok
    _el = time.time() - _t_train
    rec = {'step': step + 1, 'lr': optimizer.param_groups[0]['lr'], 'loss': statistics.mean(losses),
           'losses': [round(x, 4) for x in losses], 'ids': [TRAIN[j]['id'] for j in batch],
           'grad_norm': grad_norm, 'tokens': ntok, 'sec': round(time.time() - t0, 1),
           'tok_per_s': round(ntok / (time.time() - t0), 1),
           'eta_h': round(_el / _tok_done * (_tok_total - _tok_done) / 3600, 2)}
    HISTORY.append(rec)
    with LOG_PATH.open('a', encoding='utf-8') as f:
        f.write(json.dumps(rec) + '\n')
    print(f"step {rec['step']:>3}/{TOTAL_STEPS} loss {rec['loss']:.4f} gnorm {grad_norm:.3f} lr {rec['lr']:.2e} "
          f"{rec['tok_per_s']:.0f} tok/s  eta {rec['eta_h']:.2f} h  peak {peak_gib()}")
    if SAVE_EVERY_STEPS and (step + 1) % SAVE_EVERY_STEPS == 0 and step + 1 < TOTAL_STEPS:
        write_adapter(CKPT_DIR)
        (CKPT_DIR / 'step.txt').write_text(str(step + 1))
        eval_and_pick(step + 1)
STEPS_DONE = len(HISTORY)
print(f'trained {STEPS_DONE}/{TOTAL_STEPS} steps in {(time.time() - _t_train) / 3600:.2f} h'
      + (f' -- {STOP_REASON}' if STOP_REASON else ''))
assert STEPS_DONE > 0, 'no optimizer step ran'
eval_and_pick(STEPS_DONE)
if PICKER.step is not None and PICKER.step != STEPS_DONE:
    restore_lora(SELECTED_LORA)
    print(f'restored the adapter of step {PICKER.step}: later steps got worse on the holdout')
SELECTED_STEP = PICKER.step if PICKER.step is not None else STEPS_DONE
del SELECTED_LORA


## 8. 训练后探针（Gate 1 的 HF 侧）

- base 与 LoRA 的 `max |Δlogprob|` 应明显大于噪声。
- 关掉 LoRA 后应与 base 完全一致，用来确认 base 权重没被改动。
- 留出题上结束思考和发起工具调用的 logprob 变化，以及 `max|lora_B|` 是否仍然贴着 Adam 移动上限。这些只是 HF 侧的信号，最终以 dev-eval 的 Gate 2 为准。


In [ ]:
# TASK.md Gate 1, HF side: the adapter must change the policy, and switching it off must
# give back the base exactly. (vLLM loading and /v1/models are checked in the dev-eval notebook.)
LORA_LP = probe(PROBE_ROWS)
set_lora(False)
OFF_LP = probe(PROBE_ROWS)
set_lora(True)

_d_lora = max((a - b).abs().max().item() for a, b in zip(LORA_LP, BASE_LP))
_d_off = max((a - b).abs().max().item() for a, b in zip(OFF_LP, BASE_LP))
PROBE_RESULT = {
    'rows': [r['id'] for r in PROBE_ROWS],
    'base_nll': BASE_NLL,
    'lora_nll': [round(-lp.mean().item(), 4) for lp in LORA_LP],
    'max_abs_dlogprob_lora_vs_base': round(_d_lora, 4),
    'max_abs_dlogprob_off_vs_base': round(_d_off, 6),
    'mean_abs_dlogprob_lora_vs_base': round(statistics.mean((a - b).abs().mean().item() for a, b in zip(LORA_LP, BASE_LP)), 5),
}
_max_b = max(m.lora_B.detach().abs().max().item() for m in LORAS.values())
_bound = adam_movement_bound(TOTAL_STEPS, WARMUP_STEPS, LR)
HOLDOUT_FINAL = holdout_eval()
PROBE_RESULT.update(max_abs_lora_B=round(_max_b, 6), adam_movement_bound=round(_bound, 6),
                    selected_step=SELECTED_STEP, holdout_base=HOLDOUT_BASE, holdout_final=HOLDOUT_FINAL)
print(json.dumps(PROBE_RESULT, indent=1))
if _max_b >= 0.9 * _bound:
    print(f'NOTE: max|lora_B| {_max_b:.5f} is at the Adam movement bound {_bound:.5f}: training is still limited by steps x lr.')
if HOLDOUT_BASE and HOLDOUT_FINAL and HOLDOUT_BASE['channel_end_lp'] is not None:
    _dch = HOLDOUT_FINAL['channel_end_lp'] - HOLDOUT_BASE['channel_end_lp']
    print(f"holdout: weighted NLL {HOLDOUT_BASE['weighted_nll']} -> {HOLDOUT_FINAL['weighted_nll']}, "
          f"<channel|> logprob change {_dch:+.4f}")
    if _dch <= 0 or HOLDOUT_FINAL['weighted_nll'] >= HOLDOUT_BASE['weighted_nll']:
        print('WARNING: no improvement on the holdout tasks: this adapter is unlikely to shorten inference on new tasks.')
assert _d_off < 1e-2, f'LoRA off differs from the base by {_d_off}: the base weights changed during training'
if _d_lora < 5 * max(_d_off, 1e-3):
    print('WARNING: the adapter barely changes the probe log-probs. Gate 1 needs max |dlogprob| well above '
          'the base noise floor: raise LR / train longer before submitting.')


## 9. 导出 adapter + key 检查

- **导出**：标准 PEFT 格式，bf16，不 merge。key 用 checkpoint 自己的模块名（`base_model.model.{P}layers.{i}.self_attn.{q,k,v,o}_proj.lora_{A,B}.weight`），vLLM 会对权重和 LoRA 用同一套名字映射。
- **检查**：
  - 每个模块都是 checkpoint 中的 int4 Linear。
  - A 为 `[r, in]`，B 为 `[out, r]`，dtype 为 bf16。
  - `target_modules` 覆盖所有模块。

In [ ]:
from safetensors import safe_open

ADAPTER_DIR = WORKING_DIR / ADAPTER_NAME
write_adapter(ADAPTER_DIR)

# Gate 1 key check against the checkpoint vLLM serves: every LoRA module must be an int4
# Linear of the checkpoint (same name, so vLLM's own name mapping applies to both), with
# A = [r, in], B = [out, r], bf16, rank r everywhere.
ADAPTER_MODULES, _problems = check_adapter(ADAPTER_DIR, CKPT_LINEAR_SHAPES, LORA_R)
assert not _problems, 'adapter check failed:\n  ' + '\n  '.join(_problems[:30])
assert all(m + '.weight_packed' in WEIGHT_MAP for m in ADAPTER_MODULES)
assert all(m.startswith(CKPT_PREFIX + 'layers.') for m in ADAPTER_MODULES), 'LoRA outside the language-model layers'

# the file holds exactly the trained weights (bf16)
with safe_open(str(ADAPTER_DIR / 'adapter_model.safetensors'), 'pt') as f:
    for name, m in list(LORAS.items())[:8]:
        k = 'base_model.model.' + to_ckpt(name)
        assert torch.equal(f.get_tensor(k + '.lora_B.weight'), m.lora_B.detach().cpu().to(torch.bfloat16))
_b_norm = statistics.mean(m.lora_B.detach().norm().item() for m in LORAS.values())
print(f'{ADAPTER_DIR}: {len(ADAPTER_MODULES)} modules, r={LORA_R}, alpha={LORA_ALPHA}, targets {LORA_TARGETS}, '
      f'{(ADAPTER_DIR / "adapter_model.safetensors").stat().st_size / 1024**2:.1f} MiB, mean |B| {_b_norm:.4f}')
print('example key:', next(iter(sorted(ADAPTER_MODULES))))
print(json.dumps(json.loads((ADAPTER_DIR / 'adapter_config.json').read_text()), indent=1))

## 10. 打包 submission.zip

- 复制挂载的 submission，加入 `adapters/main_lora/`，并在 `agent.yaml` 的 `model:` 下一行插入 `adapter: main_lora`（按文本编辑，保留 `!include`）。
- 用 `scripts/package_submission.py` 的规则校验：
  - 扩展名
  - 软链接
  - < 3 GiB
  - 时间预算：`max_time_minutes × 120 ≤ 10.5 h`，且 prompt 中的分钟数与之一致
- 装了 harness 时，再跑它的 `validate_directory` / `discover_adapters`。
- 时间预算不通过、或 submission 与采样时不一致，都不会生成 zip（SMOKE 只打印警告）。
- `ALPHA_VARIANTS` 中每个额外的 alpha 都会另外打包一个 `submission_alpha{a}.zip`。


In [ ]:
def _zip_has_root_agent(z: Path) -> bool:
    try:
        with zipfile.ZipFile(z) as zf:
            return 'agent.yaml' in zf.namelist()
    except zipfile.BadZipFile:
        return False


def _not_a_submission(p: Path) -> bool:
    """Copies inside other attached outputs: the data notebook's submission_src/, sample dirs."""
    return any(part == 'submission_src' or part.startswith(('rft_samples_', 'rft_data_', 'rft_slots')) for part in p.parts)


def find_submission() -> Path:
    """Same as the data notebook: a zip with agent.yaml at its root, or a directory holding agent.yaml."""
    if SUBMISSION_INPUT:
        roots = [Path(SUBMISSION_INPUT)]
    else:
        roots = [p for p in (INPUT_DIR / 'datasets').glob('*/*') if p.is_dir()]
        roots += [p for p in INPUT_DIR.iterdir() if p.is_dir() and p.name not in ('competitions', 'models', 'datasets')]
        roots = [r for r in roots if r.resolve() != WHEELHOUSE_DIR.resolve()]
    zips, dirs = [], []
    for root in roots:
        if root.is_file():
            zips += [root] if _zip_has_root_agent(root) else []
            continue
        zips += [z for z in root.rglob('*.zip') if not _not_a_submission(z) and _zip_has_root_agent(z)]
        dirs += [p.parent for p in root.rglob('agent.yaml') if not _not_a_submission(p)]
    candidates = zips or dirs
    if len(candidates) != 1:
        raise RuntimeError(f'Expected exactly one submission, found zips={zips} dirs={dirs}. '
                           'Attach your submission dataset, or set SUBMISSION_INPUT.')
    return candidates[0]


SUBMISSION_SRC = find_submission()
OUT_SUBMISSION = WORKING_DIR / 'submission'
if OUT_SUBMISSION.exists():
    shutil.rmtree(OUT_SUBMISSION)
if SUBMISSION_SRC.suffix == '.zip':
    with zipfile.ZipFile(SUBMISSION_SRC) as zf:
        zf.extractall(OUT_SUBMISSION)
else:
    shutil.copytree(SUBMISSION_SRC, OUT_SUBMISSION)
_base_files, _ = collect_submission(OUT_SUBMISSION)
BASE_SUBMISSION_FINGERPRINT = submission_fingerprint(OUT_SUBMISSION, _base_files)
print(f'submission source: {SUBMISSION_SRC} (fingerprint {BASE_SUBMISSION_FINGERPRINT}; '
      f'RFT data was sampled with {DATA_STATS.get("submission_fingerprint")})')
if BASE_SUBMISSION_FINGERPRINT != DATA_STATS.get('submission_fingerprint'):
    _msg = ('this submission differs from the one the RFT data was sampled with: the adapter is trained on that '
            'prompt / tool set. Attach the same submission, or set ALLOW_SUBMISSION_MISMATCH = True.')
    if not (SMOKE or ALLOW_SUBMISSION_MISMATCH):
        raise RuntimeError(_msg)
    print('WARNING:', _msg)
if (OUT_SUBMISSION / 'adapters').exists():
    print('WARNING: the attached submission already has adapters/; they are replaced by this adapter.')
    shutil.rmtree(OUT_SUBMISSION / 'adapters')

_agent_yaml = OUT_SUBMISSION / 'agent.yaml'
_agent_yaml.write_text(add_adapter_to_agent_yaml(_agent_yaml.read_text(encoding='utf-8'), ADAPTER_NAME), encoding='utf-8')
_eval_cfg = OUT_SUBMISSION / 'eval_config.yaml'
print('--- agent.yaml\n' + _agent_yaml.read_text(encoding='utf-8'))
if _eval_cfg.is_file():
    print('--- eval_config.yaml (unchanged)\n' + _eval_cfg.read_text(encoding='utf-8'))
_time_problems = check_time_budget(OUT_SUBMISSION)
if _time_problems:
    _msg = 'time budget check failed:\n  ' + '\n  '.join(_time_problems)
    if not SMOKE:
        raise RuntimeError(_msg)   # an overrun errors the whole submission
    print('WARNING:', _msg)
else:
    print(f'time budget OK: worst case at most {MAX_WORST_CASE_HOURS} h for {N_HIDDEN_TASKS} tasks')


def package(alpha: float, zip_path: Path) -> str:
    adapter_dst = OUT_SUBMISSION / 'adapters' / ADAPTER_NAME
    if adapter_dst.exists():
        shutil.rmtree(adapter_dst)
    shutil.copytree(ADAPTER_DIR, adapter_dst)
    if alpha != LORA_ALPHA:   # inference-time interpolation between base and LoRA (TASK.md §3.2)
        c = json.loads((adapter_dst / 'adapter_config.json').read_text())
        c['lora_alpha'] = alpha
        (adapter_dst / 'adapter_config.json').write_text(json.dumps(c, indent=2) + '\n')
    # the harness's own checks, when its packages are installed (wheelhouse)
    try:
        from adk_submission import discover_adapters, validate_directory
        from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS, build_submission_limits
    except ImportError as e:
        print(f'  (harness packages not importable, skipping its checks: {e})')
    else:
        validate_directory(OUT_SUBMISSION, limits=build_submission_limits()[0])
        found = getattr(discover_adapters(str(OUT_SUBMISSION), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS), 'adapters', None) or {}
        assert ADAPTER_NAME in found, f'harness did not discover adapter {ADAPTER_NAME}: {sorted(found)}'
        rank = getattr(found[ADAPTER_NAME], 'rank', None)
        assert rank in (None, LORA_R), f'harness sees rank {rank}, expected {LORA_R}'
        print(f'  harness: validate_directory OK, adapter {ADAPTER_NAME!r} discovered (rank {rank})')
    files, fp = write_submission_zip(OUT_SUBMISSION, zip_path)
    print(f'{zip_path} ({zip_path.stat().st_size / 1024**2:.1f} MiB, {len(files)} files, fingerprint {fp}, lora_alpha {alpha})')
    for p in files:
        print('   ', p.relative_to(OUT_SUBMISSION).as_posix())
    return fp


_suffix = '_SMOKE' if SMOKE else ''
PACKAGES = {}
for a in [x for x in ALPHA_VARIANTS if x != LORA_ALPHA]:
    PACKAGES[a] = (f'submission{_suffix}_alpha{a:g}.zip', package(a, WORKING_DIR / f'submission{_suffix}_alpha{a:g}.zip'))
PACKAGES[LORA_ALPHA] = (f'submission{_suffix}.zip', package(LORA_ALPHA, WORKING_DIR / f'submission{_suffix}.zip'))  # last: OUT_SUBMISSION holds the main one

TRAIN_META = {
    'run_name': RUN_NAME,
    'smoke': SMOKE,
    'created_at_utc': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime()),
    'model_path': str(MODEL_PATH),
    'checkpoint_prefix': CKPT_PREFIX,
    'quantization': _wargs,
    'versions': {'torch': torch.__version__, 'transformers': transformers.__version__},
    'data': DATA_SUMMARY,
    'hparams': {'r': LORA_R, 'lora_alpha': LORA_ALPHA, 'dropout': LORA_DROPOUT, 'targets': LORA_TARGETS,
                'n_lora_modules': len(LORAS), 'lr': LR, 'warmup_ratio': WARMUP_RATIO, 'epochs': EPOCHS,
                'grad_accum': GRAD_ACCUM, 'max_grad_norm': MAX_GRAD_NORM, 'weight_decay': WEIGHT_DECAY, 'seed': SEED,
                'max_loss_tokens': MAX_LOSS_TOKENS, 'max_seq_tokens': MAX_SEQ_TOKENS,
                'decision_weight': DECISION_WEIGHT, 'first_edit_weight': FIRST_EDIT_WEIGHT,
                'loss': 'weighted mean per sequence, then weighted mean over the batch'},
    'layers_per_gpu': dict(Counter(LAYER_DEV)),
    'steps': {'done': STEPS_DONE, 'total': TOTAL_STEPS, 'stop_reason': STOP_REASON, 'selected': SELECTED_STEP},
    'holdout_history': HOLDOUT_HISTORY,
    'loss_first_last': [HISTORY[0]['loss'], HISTORY[-1]['loss']],
    'history': HISTORY,
    'probe': PROBE_RESULT,
    'submission': {'source': str(SUBMISSION_SRC), 'base_fingerprint': BASE_SUBMISSION_FINGERPRINT,
                   'rft_data_fingerprint': DATA_STATS.get('submission_fingerprint'),
                   'packages': {str(a): {'zip': z, 'fingerprint': fp} for a, (z, fp) in PACKAGES.items()}},
    'notebook_hours': round((time.time() - NOTEBOOK_START) / 3600, 2),
}
(WORKING_DIR / 'train_meta.json').write_text(json.dumps(TRAIN_META, indent=2, default=str), encoding='utf-8')

print(f"\nloss {HISTORY[0]['loss']:.4f} -> {HISTORY[-1]['loss']:.4f} over {STEPS_DONE}/{TOTAL_STEPS} steps; "
      f"probe NLL {PROBE_RESULT['base_nll']} -> {PROBE_RESULT['lora_nll']}, "
      f"max |dlogprob| {PROBE_RESULT['max_abs_dlogprob_lora_vs_base']}")
print(f'notebook time: {TRAIN_META["notebook_hours"]} h')
if SMOKE:
    print('SMOKE run: submission_SMOKE.zip only checks the pipeline. Set SMOKE = False for the real adapter.')
else:
    print('Next: attach submission.zip to the dev-eval notebook -> Gate 1 (vLLM loads the adapter, /v1/models), '
          'then Gate 2a/2b (TASK.md §4).')
